# Notebook 6 — Fine-tuned Qwen3-8B (`Qwen3-8B-PM-LoRA`) on **Real** PM Data

**Question:** does the adapter that was trained on synthetic data (Notebook 4) also help on **real Apache Jira** tasks?

**Model added:** `Qwen3-8B-FT` = `Qwen/Qwen3-8B` + LoRA adapter [`arjunverma2004/Qwen3-8B-PM-LoRA`](https://huggingface.co/arjunverma2004/Qwen3-8B-PM-LoRA)
**Compared with:** the 4 models of Notebook 5 (Qwen3-8B · Llama-3.1-8B · GLM-Z1-9B · Gemini-3.5-Flash-Lite)
**Data / metrics / judge:** exactly the same 200 real samples, real-track metrics and judge as Notebook 5

---

### How this notebook is organised

This notebook is a **copy of Notebook 5** with one extra cell at the end:

| Cells | Content | Where to read the details |
|---|---|---|
| 1–8 | setup, data loader, inference engines, metrics, judge, runner, main run (4 base models, from checkpoints) | **Notebook 5** — same code, same outputs |
| 9–12 | per-track tables, tests, paper tables, charts — **run before the FT model was added**, so they show 4 models | Notebook 5 |
| **13 (new)** | **load the FT model → generate → score → judge → merge as the 5th model** | this notebook |

The analysis of the merged 5-model results (tables, tests, charts) is in **Notebook 7**, which recomputes it from
`ft-real_benchmark_results_real_001.csv` — the file this notebook writes.

### Results at a glance (computed from the saved CSV; full tables in Notebook 7)

| Track | Qwen3-8B (base) | **Qwen3-8B-FT** | What changed |
|---|---|---|---|
| Task design — judge / schema | 2.13 / 0.98 | **2.68 / 1.00** | ✅ clear improvement, close to Llama (2.69) and Gemini (2.82) |
| Task allocation — judge | 1.26 | 1.17 | ≈ no change (both hit the prompt-truncation bug) |
| Sprint planning — judge / task-set F1 / valid JSON | 2.41 / 0.27 / 58 % | **1.86 / 0.00 / 0 %** | ❌ worse — 45 of 50 FT answers are cut off mid-JSON at 600 tokens |
| Resource planning — judge | 1.55 | 1.53 | ≈ no change |
| Overall judge (all judged rows) | 1.81 | 1.83 | ≈ no change |
| Latency per sample | 3.66 s | **1.05 s** | ✅ faster (bf16 + batch 32 + no thinking); tok/s is not comparable, see Cell 13 note |

**Interpretation.** Fine-tuning on synthetic data helps where the real task looks like the synthetic one (task design)
and hurts where the real task asks for a **different output format** (sprint planning: real data asks for
`scope_size` and a list of issue titles; the FT model writes the synthetic format — objects with `id`, `title`,
`story_points`, `assignee`, `notes` — which runs past 600 tokens and leaves invalid JSON).

### Relationship to the other notebooks

| Notebook | Role |
|---|---|
| 4 · fine-tune + synthetic eval | trained and published the adapter used here |
| 5 · real-data benchmark | the 4 base models on the same 200 real samples |
| **6 · this notebook** | adds the fine-tuned model as a 5th model |
| 7 · final comparison | 5-model real-data tables, tests and charts; synthetic vs real |

> Code and outputs are unchanged from the original run (keys replaced with placeholders).

## Cells 1–2 — Install & Configuration

Same as Notebook 5, plus **`peft`** (needed to load the LoRA adapter in Cell 13). Configuration is identical
(same models, same 200-sample real dataset, same generation settings, same checkpoint folder) — that is what makes
the FT results directly comparable with the base models.

In [1]:
!pip install transformers accelerate torch \
    bitsandbytes flash-attn --quiet
!pip install rouge_score bert_score --quiet
!pip install google-genai --quiet
# NEW vs Notebook 5: PEFT loads the LoRA adapter.
!pip install peft --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os

# Placeholders — use your own keys.
# ── API Keys ─────────────────────────────────────────────────────────────────
os.environ["INCEPTION_API_KEY"] = "YOUR_INCEPTION_API_KEY_1"
os.environ["HF_TOKEN"]          = "YOUR_HF_TOKEN"
os.environ["GEMINI_API_KEY"]    = "YOUR_GEMINI_API_KEY"

# ── Local model registry (unchanged from v5) ─────────────────────────────────
MODELS = {
    "Qwen3-8B":      "Qwen/Qwen3-8B",
    "GLM-Z1-9B":     "zai-org/GLM-Z1-9B-0414",
    "Llama-3.1-8B":  "meta-llama/Llama-3.1-8B-Instruct",
}

# ── API model registry (new — no local load/unload, called over HTTP) ────────
API_MODELS = {
    "Gemini-3.5-Flash-Lite": "gemini-3.5-flash-lite",
}

# ── REAL dataset paths ────────────────────────────────────────────────────────
REAL_DATASET_PATHS = {
    "task_design":       "real_task_design.json",
    "task_allocation":   "real_task_allocation.json",
    "sprint_planning":   "real_sprint_planning_v3.json",
    "resource_planning": "real_resource_planning_v3.json",
}
SAMPLES_PER_TYPE = 50   # 50 × 4 types = 200 total, sampled from each real file  # same 50 per track as Notebook 5
RANDOM_SEED      = 42   # fixed seed → same 200 samples every re-run  # same seed → the FT model gets the same 200 prompts

# ── Gemini API rate limit ─────────────────────────────────────────────────────
GEMINI_REQUESTS_PER_MIN = 11   # hard cap — 60/11 ≈ 5.45s between calls
GEMINI_SLEEP_BETWEEN    = 60.0 / GEMINI_REQUESTS_PER_MIN

# ── Inference settings (unchanged) ────────────────────────────────────────────
MAX_NEW_TOKENS  = 600  # same budget for every model (important for the sprint-planning result)
TEMPERATURE     = 0.3
DO_SAMPLE       = True
BATCH_SIZE      = 12
USE_4BIT        = True

# ── Judge ─────────────────────────────────────────────────────────────────────
JUDGE_MODEL   = "mercury-2"
INCEPTION_URL = "https://api.inceptionlabs.ai/v1/chat/completions"

# ── Output ────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "checkpoints_real"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("✅ Config loaded")
print(f"Local models : {list(MODELS.keys())}")
print(f"API models   : {list(API_MODELS.keys())}")
print(f"Samples      : {SAMPLES_PER_TYPE * 4} total ({SAMPLES_PER_TYPE} per task type)")

✅ Config loaded
Local models : ['Qwen3-8B', 'GLM-Z1-9B', 'Llama-3.1-8B']
API models   : ['Gemini-3.5-Flash-Lite']
Samples      : 200 total (50 per task type)


## Cell 3 — Real Dataset Loader (same as Notebook 5)

Fixed seed 42 → the same 200 samples (50 per track) that the 4 base models answered.

In [11]:
import json, random
import pandas as pd

random.seed(RANDOM_SEED)

VALID_TASK_TYPES = {
    "task_design",
    "task_allocation",
    "sprint_planning",
    "resource_planning",
}

# Real-track required fields differ slightly from the synthetic schema
# (e.g. sprint_planning uses scope_size, not total_points — story points
# weren't populated on any ASF project pulled, see Methodology note).
REQUIRED_FIELDS = {
    "task_design":       ["title", "description", "acceptance_criteria", "story_points", "priority"],
    "task_allocation":   ["assigned_to", "role", "rationale", "estimated_hours"],
    "sprint_planning":   ["sprint_goal", "selected_tasks", "scope_size", "excluded_tasks"],
    "resource_planning": ["capacity_constraints", "risk_flags", "mitigation_recommendations"],
}


# Same loader as Notebook 5 (see there for details).
def load_real_dataset(paths: dict, samples_per_type: int, seed: int) -> list:
    dataset = []
    print("📊 Real dataset breakdown:")
    for task_type, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Real dataset not found: {path}")
        with open(path, "r", encoding="utf-8") as f:
            samples = json.load(f)

        for s in samples:
            s["task_type"] = task_type
            if "reference_output" not in s:
                s["reference_output"] = ""

        rng = random.Random(seed)
        selected = samples if len(samples) <= samples_per_type else rng.sample(samples, samples_per_type)
        dataset.extend(selected)

        status = "✅" if len(samples) >= samples_per_type else "⚠️ "
        print(f"   {status} {task_type}: {len(selected)} used / {len(samples)} available")

    print(f"\n✅ Total REAL samples loaded: {len(dataset)}")
    return dataset


REAL_BENCHMARK_DATASET = load_real_dataset(REAL_DATASET_PATHS, SAMPLES_PER_TYPE, RANDOM_SEED)

📊 Real dataset breakdown:
   ✅ task_design: 50 used / 360 available
   ✅ task_allocation: 50 used / 52 available
   ✅ sprint_planning: 50 used / 71 available
   ✅ resource_planning: 50 used / 71 available

✅ Total REAL samples loaded: 200


## Cell 4 — Local 4-bit Inference Engine (same as Notebook 5)

Used for the 3 base SLMs. The FT model does **not** use this loader — Cell 13 defines its own bf16 loader.

> ⚠️ The same `max_length=1024` prompt truncation applies to the FT engine in Cell 13 (see Notebook 5, Cell 4 note).

In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import time
import gc
from tqdm import tqdm


def get_bnb_config() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )


# 4-bit loader for the base SLMs (Notebook 5, Cell 4).
def load_model_4bit(model_name: str, model_id: str):
    print(f"\n{'='*60}")
    print(f"⏳ Loading {model_name}")
    print(f"   HF ID : {model_id}")
    print(f"{'='*60}")

    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, token=os.environ.get("HF_TOKEN"), trust_remote_code=True,
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = get_bnb_config() if USE_4BIT else None
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
        token=os.environ.get("HF_TOKEN"),
        trust_remote_code=True,
        attn_implementation="flash_attention_2",
    )
    model.eval()

    load_time = time.time() - t0
    vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"✅ {model_name} ready | Load: {load_time:.1f}s | VRAM: {vram_used:.1f} GB")
    return tokenizer, model


def build_prompt(sample: dict, model_name: str, tokenizer) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Agile project manager. "
                "When asked to output JSON, respond ONLY with valid JSON — "
                "no markdown fences, no explanations, no preamble."
            ),
        },
        {"role": "user", "content": sample["input"]},
    ]
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
    except Exception:
        formatted = (
            "System: You are an expert Agile project manager.\n\n"
            f"User: {sample['input']}\n\nAssistant:"
        )
    if "Qwen3" in model_name or "qwen3" in model_name.lower():
        formatted = formatted.replace("<|im_start|>user", "<|im_start|>user /no_think", 1)
    return formatted


def run_batched_inference(tokenizer, model, dataset: list, model_name: str, batch_size: int = BATCH_SIZE) -> list:
    results = []
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    print(f"\n🚀 Running batched inference: {len(dataset)} samples, batch_size={batch_size}")

    prompts = [build_prompt(s, model_name, tokenizer) for s in dataset]
    total_tokens = 0
    wall_t0 = time.time()

    for batch_idx in tqdm(range(n_batches), desc=f"{model_name}", ncols=80):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(dataset))
        batch_prompts = prompts[start:end]
        batch_samples = dataset[start:end]

        # ⚠️ long prompts lose their end
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[1]

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
                do_sample=DO_SAMPLE, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.05,
            )
        batch_latency = time.time() - t0

        for sample, output_ids in zip(batch_samples, outputs):
            generated_ids = output_ids[input_len:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            n_tokens = int((generated_ids != tokenizer.eos_token_id).sum())
            total_tokens += n_tokens

            results.append({
                "sample_id": sample["id"], "task_type": sample["task_type"],
                "input": sample["input"], "reference": sample["reference_output"],
                "response": response,
                "latency_s": round(batch_latency / len(batch_samples), 2),
                "tokens_gen": n_tokens,
                "tokens_per_sec": round(n_tokens / (batch_latency / len(batch_samples)), 1),
                # carry through real-track metadata for scoring
                "_meta": {k: v for k, v in sample.items() if k not in ("id", "task_type", "input", "reference_output")},
            })

    wall_elapsed = time.time() - wall_t0
    print(f"\n✅ {model_name} inference done | {wall_elapsed/60:.1f} min | {total_tokens:,} tokens")
    return results


def unload_model(model, tokenizer):
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"🗑️  Model unloaded | VRAM now: {torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0:.1f} GB")


print("✅ Local inference engine ready (unchanged from v5)")

✅ Local inference engine ready (unchanged from v5)


## Cell 4b — Gemini API Inference (same as Notebook 5)

> The output shows `ImportError: cannot import name 'Sentinel' from 'typing_extensions'` — a version conflict
> introduced by installing `peft` in this Colab session. It is harmless here because Gemini's answers are loaded
> from the checkpoint in Cell 8 and no Gemini call is made. To silence it: `pip install -U typing_extensions` and restart.

In [5]:
import os
import time as _time
from google import genai  # ⚠️ fails in this session (typing_extensions conflict) — not needed, Gemini comes from its checkpoint
from google.genai import types
from tqdm import tqdm

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

GEMINI_SYSTEM_PROMPT = (
    "You are an expert Agile project manager. "
    "When asked to output JSON, respond ONLY with valid JSON — "
    "no markdown fences, no explanations, no preamble."
)


def run_gemini_inference(dataset: list, model_id: str, model_name: str,
                          max_retries: int = 3, sleep_between: float = None) -> list:
    """
    Sequential API inference (no local batching — API has its own
    concurrency handling). Retries on transient errors, matches the same
    result schema as run_batched_inference so it merges cleanly.
    """
    sleep_between = GEMINI_SLEEP_BETWEEN if sleep_between is None else sleep_between
    
    # Define generation parameters using the new structured configuration schema
    config = types.GenerateContentConfig(
        system_instruction=GEMINI_SYSTEM_PROMPT,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_NEW_TOKENS,
    )

    results = []
    print(f"\n🚀 Running Gemini inference: {len(dataset)} samples "
          f"| rate limit: {GEMINI_REQUESTS_PER_MIN} req/min ({sleep_between:.2f}s between calls)")
    for sample in tqdm(dataset, desc=model_name, ncols=80):
        response_text, latency = "", 0.0
        for attempt in range(max_retries):
            t0 = _time.time()
            try:
                # Use client.models.generate_content for the new SDK structure
                resp = client.models.generate_content(
                    model=model_id,
                    contents=sample["input"],
                    config=config
                )
                response_text = (resp.text or "").strip()
                latency = _time.time() - t0
                break
            except Exception as e:
                latency = _time.time() - t0
                if attempt == max_retries - 1:
                    print(f"\n⚠️  Gemini failed on {sample['id']}: {e}")
                    response_text = ""
                else:
                    _time.sleep(2 * (attempt + 1))

        results.append({
            "sample_id": sample["id"], "task_type": sample["task_type"],
            "input": sample["input"], "reference": sample["reference_output"],
            "response": response_text,
            "latency_s": round(latency, 2),
            "tokens_gen": None,       # not exposed uniformly by the API
            "tokens_per_sec": None,
            "_meta": {k: v for k, v in sample.items() if k not in ("id", "task_type", "input", "reference_output")},
        })
        _time.sleep(sleep_between)  # be polite to the API rate limit

    print(f"\n✅ Gemini inference done")
    return results


print("✅ Gemini API inference ready")


ImportError: cannot import name 'Sentinel' from 'typing_extensions' (/usr/local/lib/python3.12/site-packages/typing_extensions.py)

## Cell 5 — Real-Track Metrics Engine (same as Notebook 5)

The FT model is scored by **exactly the same** `compute_real_metrics` as the 4 base models.
See Notebook 5, Cell 5 for the metric definitions and their caveats (priority baseline, velocity MAPE, hours outliers).

In [6]:
import re, difflib
from rouge_score import rouge_scorer  # kept for optional diagnostic use, not scored into composites


def extract_json(text: str):
    clean = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        pass
    match = re.search(r"(\{.*\}|\[.*\])", clean, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    return None


def check_json_validity(response: str, task_type: str) -> dict:
    parsed = extract_json(response)
    if parsed is None:
        return {"valid_json": False, "schema_compliance": 0.0,
                "missing_fields": REQUIRED_FIELDS.get(task_type, [])}
    required = REQUIRED_FIELDS.get(task_type, [])
    check_obj = parsed[0] if isinstance(parsed, list) and parsed else parsed
    if not isinstance(check_obj, dict):
        return {"valid_json": True, "schema_compliance": 0.0, "missing_fields": required}
    present = [f for f in required if f in check_obj]
    missing = [f for f in required if f not in check_obj]
    return {"valid_json": True,
            "schema_compliance": round(len(present) / len(required), 3) if required else 1.0,
            "missing_fields": missing}


HALLUC_PATTERNS = [
    r"I (don't|do not|cannot|can't) (know|provide|access)",
    r"As an AI", r"I apologize", r"I'm sorry",
]

def detect_hallucination(response: str) -> bool:
    return any(re.search(p, response, re.IGNORECASE) for p in HALLUC_PATTERNS)


def _fuzzy_match(a: str, b: str, threshold: float = 0.6) -> bool:
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio() >= threshold


def _first_number(text: str):
    m = re.search(r"[-+]?\d*\.?\d+", text or "")
    return float(m.group(0)) if m else None


def _mape(pred, real):
    if pred is None or real in (None, 0):
        return None
    return round(abs(pred - real) / abs(real), 4)


# ── Track-specific scorers ────────────────────────────────────────────────

def score_task_design(parsed: dict, meta: dict) -> dict:
    pred_priority = str((parsed or {}).get("priority", "")).strip().capitalize()
    known = meta.get("known_priority")
    return {
        "priority_predicted": pred_priority,
        "priority_known": known,
        "priority_match": bool(known) and pred_priority == known,
    }


def score_task_allocation(parsed: dict, meta: dict) -> dict:
    pred_assignee = str((parsed or {}).get("assigned_to", "")).strip()
    pred_hours    = (parsed or {}).get("estimated_hours")
    try:
        pred_hours = float(pred_hours) if pred_hours is not None else None
    except (TypeError, ValueError):
        pred_hours = None

    real_assignee = meta.get("ground_truth_assignee")
    real_hours    = meta.get("ground_truth_hours")

    exact_match = bool(real_assignee) and pred_assignee.strip().lower() == real_assignee.strip().lower()
    # plausible match: best-effort — same as exact unless a roster/skill lookup
    # is wired in; exact match is the primary metric, this is reported alongside.
    plausible_match = exact_match

    return {
        "assignee_predicted": pred_assignee,
        "assignee_real": real_assignee,
        "assignee_exact_match": exact_match,
        "assignee_plausible_match": plausible_match,
        "hours_predicted": pred_hours,
        "hours_real": real_hours,
        "hours_mape": _mape(pred_hours, real_hours),
    }


def score_sprint_planning(parsed: dict, meta: dict) -> dict:
    pred_tasks = (parsed or {}).get("selected_tasks", [])
    if not isinstance(pred_tasks, list):
        pred_tasks = []
    pred_tasks = [str(t) for t in pred_tasks]

    real_ref = meta.get("reference_output_parsed") or {}
    real_tasks = real_ref.get("selected_tasks", [])
    real_scope = real_ref.get("scope_size")

    matched = 0
    used = set()
    for pt in pred_tasks:
        for i, rt in enumerate(real_tasks):
            if i in used:
                continue
            if _fuzzy_match(pt, rt):
                matched += 1
                used.add(i)
                break

    precision = matched / len(pred_tasks) if pred_tasks else 0.0
    recall    = matched / len(real_tasks) if real_tasks else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    pred_scope = (parsed or {}).get("scope_size")
    try:
        pred_scope = float(pred_scope) if pred_scope is not None else None
    except (TypeError, ValueError):
        pred_scope = None

    return {
        "taskset_precision": round(precision, 3),
        "taskset_recall": round(recall, 3),
        "taskset_f1": round(f1, 3),
        "scope_predicted": pred_scope,
        "scope_real": real_scope,
        "scope_mape": _mape(pred_scope, real_scope),
    }


def score_resource_planning(parsed: dict, meta: dict) -> dict:
    capacity_text = str((parsed or {}).get("capacity_constraints", ""))
    pred_velocity = _first_number(capacity_text)  # best-effort, low-confidence

    real_ref = meta.get("reference_output_parsed") or {}
    real_velocity = real_ref.get("real_avg_velocity")

    return {
        "velocity_predicted_bestguess": pred_velocity,
        "velocity_real": real_velocity,
        "velocity_mape": _mape(pred_velocity, real_velocity),
        "velocity_confidence": "low — extracted from free text, not a dedicated field",
    }


TRACK_SCORERS = {
    "task_design": score_task_design,
    "task_allocation": score_task_allocation,
    "sprint_planning": score_sprint_planning,
    "resource_planning": score_resource_planning,
}


# Shared scorer — reused unchanged for the FT model in Cell 13.
def compute_real_metrics(result: dict) -> dict:
    response  = result["response"]
    task_type = result["task_type"]
    meta      = dict(result.get("_meta", {}))

    # pre-parse the real reference_output JSON string, if present, for the
    # track scorers that need structured real fields (sprint/resource)
    ref_raw = result.get("reference", "")
    try:
        meta["reference_output_parsed"] = json.loads(ref_raw) if ref_raw else {}
    except json.JSONDecodeError:
        meta["reference_output_parsed"] = {}

    json_check = check_json_validity(response, task_type)
    parsed     = extract_json(response)
    halluc     = detect_hallucination(response)

    track_metrics = TRACK_SCORERS[task_type](parsed if isinstance(parsed, dict) else {}, meta)

    return {
        "auto_valid_json":        json_check["valid_json"],
        "auto_schema_compliance": json_check["schema_compliance"],
        "auto_missing_fields":    json_check["missing_fields"],
        "auto_hallucination":     halluc,
        **track_metrics,
    }


print("✅ Real-track metrics engine ready")

✅ Real-track metrics engine ready


## Cell 6 — LLM Judge (same as Notebook 5)

Same prompt, same truncation (answer cut to 500 characters), same keys. Reused for the FT model in Cell 13.

In [7]:
import requests
import copy
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

INCEPTION_API_KEYS = [
    "YOUR_INCEPTION_API_KEY_1",
    "YOUR_INCEPTION_API_KEY_2",
    "YOUR_INCEPTION_API_KEY_3",
]

REQUESTS_PER_MIN = 60
SLEEP_BETWEEN_CALLS = 60.0 / REQUESTS_PER_MIN

JUDGE_SYSTEM = (
    "You are a strict JSON-only evaluator. "
    "You MUST respond with a single valid JSON object and absolutely nothing else. "
    "No markdown fences, no preamble, no explanation, no trailing text. "
    "Only the raw JSON object starting with { and ending with }."
)

JUDGE_PROMPT_TEMPLATE = """\
Evaluate this AI-generated project management output.

TASK TYPE: {task_type}

INPUT:
{input_prompt}

{reference_section}

MODEL OUTPUT:
{model_output}

Score 1-5 on each dimension (1=very poor, 5=excellent).
Return ONLY this JSON object with integer scores filled in:
{{"relevance": 0, "completeness": 0, "feasibility": 0, "clarity": 0, "agile_alignment": 0, "overall_score": 0.0, "strengths": "", "weaknesses": "", "verdict": ""}}
"""

_JUDGE_NULL = {
    "relevance": None, "completeness": None, "feasibility": None,
    "clarity": None, "agile_alignment": None, "overall_score": None,
    "strengths": "JUDGE_ERROR", "weaknesses": "JUDGE_ERROR", "verdict": "ERROR",
}


def _sanitize(text: str, limit: int) -> str:
    return (text or "")[:limit].replace("{", "(").replace("}", ")")


def _parse_judge_response(content: str) -> dict:
    clean = re.sub(r"```(?:json)?\s*", "", content).replace("```", "").strip()
    for fn in [
        lambda s: json.loads(s),
        lambda s: json.loads(re.search(r"\{[^{}]*\}", s, re.DOTALL).group(0)),
        lambda s: json.loads(s[s.find("{"):s.rfind("}")+1]),
        lambda s: json.loads(re.sub(r",\s*([}\]])", r"\1", s[s.find("{"):s.rfind("}")+1])),
    ]:
        try:
            return fn(clean)
        except Exception:
            pass
    return None


def _judge_one(result: dict, api_key: str, retries: int = 3) -> dict:
    ref = result.get("reference", "")
    if ref.strip():
        reference_section = f"REFERENCE OUTPUT (partial real ground truth):\n{_sanitize(ref, 500)}"
    else:
        reference_section = (
            "REFERENCE OUTPUT: none available — this is a real-world sample with no "
            "authored reference. Grade plausibility and quality of the output given "
            "the INPUT alone."
        )

    prompt = JUDGE_PROMPT_TEMPLATE.format(
        task_type=result["task_type"],
        input_prompt=_sanitize(result["input"], 600),
        reference_section=reference_section,
        model_output=_sanitize(result["response"], 500),
    )
    payload = {
        "model": JUDGE_MODEL,
        "messages": [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
        "max_tokens": 512,
    }

    for attempt in range(retries):
        try:
            resp = requests.post(
                INCEPTION_URL,
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json=payload, timeout=30,
            )
            resp.raise_for_status()
            parsed = _parse_judge_response(resp.json()["choices"][0]["message"]["content"].strip())
            if parsed:
                return parsed
        except requests.exceptions.HTTPError:
            if resp.status_code == 429:
                backoff = 5 * (2 ** attempt)
                print(f"\n⚠️  429 on key ...{api_key[-6:]} — backing off {backoff}s")
                time.sleep(backoff)
            else:
                print(f"\n⚠️  HTTP {resp.status_code}")
        except Exception:
            time.sleep(1 * (attempt + 1))

    return copy.copy(_JUDGE_NULL)


def _worker(partition: list, api_key: str, pbar, lock, errors_counter: list):
    for result in partition:
        j = _judge_one(result, api_key)
        result["judge_relevance"]       = j.get("relevance")
        result["judge_completeness"]    = j.get("completeness")
        result["judge_feasibility"]     = j.get("feasibility")
        result["judge_clarity"]         = j.get("clarity")
        result["judge_agile_alignment"] = j.get("agile_alignment")
        result["judge_overall"]         = j.get("overall_score")
        result["judge_strengths"]       = j.get("strengths")
        result["judge_weaknesses"]      = j.get("weaknesses")
        result["judge_verdict"]         = j.get("verdict", "ERROR")
        if result["judge_verdict"] == "ERROR":
            with lock:
                errors_counter[0] += 1
        with lock:
            pbar.update(1)
        time.sleep(SLEEP_BETWEEN_CALLS)


# Reused for the FT model in Cell 13 (only rows with judge_verdict None are judged).
def run_judge_phase(all_results: list, save_every: int = 150) -> list:
    needs_judge = [r for r in all_results if r.get("judge_verdict") is None]
    n_keys = len(INCEPTION_API_KEYS)
    if not needs_judge:
        print("✅ All samples already judged.")
        return all_results

    chunk = (len(needs_judge) + n_keys - 1) // n_keys
    parts = [needs_judge[i*chunk:(i+1)*chunk] for i in range(n_keys)]
    parts = [p for p in parts if p]

    print(f"\n🧑‍⚖️  Parallel judge | {len(needs_judge)} samples | {len(parts)} keys")

    lock = threading.Lock()
    errors_counter = [0]

    with tqdm(total=len(needs_judge), desc="Judge", ncols=80) as pbar:
        with ThreadPoolExecutor(max_workers=len(parts)) as executor:
            futures = [executor.submit(_worker, part, key, pbar, lock, errors_counter)
                       for part, key in zip(parts, INCEPTION_API_KEYS)]
            saved_at = 0
            while any(not f.done() for f in futures):
                done_now = sum(1 for r in needs_judge if r.get("judge_verdict") is not None)
                if done_now - saved_at >= save_every:
                    with open(os.path.join(CHECKPOINT_DIR, f"judge_midrun_{done_now}.json"), "w") as f:
                        json.dump(all_results, f)
                    saved_at = done_now
                time.sleep(5)
            for f in futures:
                f.result()

    print(f"\n✅ Judge complete | errors: {errors_counter[0]} / {len(needs_judge)}")
    return all_results


print(f"✅ Judge ready | {len(INCEPTION_API_KEYS)} keys | {REQUESTS_PER_MIN} req/min/key")

✅ Judge ready | 3 keys | 60 req/min/key


## Cells 7–8 — Runner & Main Run (same as Notebook 5)

All 4 base models come from their checkpoints / the same run as Notebook 5; the output is identical to
Notebook 5's Cell 8 (800 rows, 155 judge errors).

In [8]:
# Same runner as Notebook 5.
def run_inference_phase(dataset: list, local_models: dict, api_models: dict) -> tuple:
    run_id = "real_001"
    all_results = []

    # ── Local (HF, 4-bit) models ──────────────────────────────────────────
    for model_name, model_id in local_models.items():
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_{run_id}.json")
        if os.path.exists(ckpt_path):
            with open(ckpt_path) as f:
                model_results = json.load(f)
            print(f"\n♻️  {model_name}: loaded {len(model_results)} results from checkpoint")
            all_results.extend(model_results)
            continue

        tokenizer, model = load_model_4bit(model_name, model_id)
        raw_results = run_batched_inference(tokenizer, model, dataset, model_name)

        model_results = []
        for raw in raw_results:
            metrics = compute_real_metrics(raw)
            model_results.append({
                "model": model_name, "sample_id": raw["sample_id"], "task_type": raw["task_type"],
                "input": raw["input"], "reference": raw["reference"], "response": raw["response"],
                "latency_s": raw["latency_s"], "tokens_gen": raw["tokens_gen"], "tokens_per_sec": raw["tokens_per_sec"],
                **metrics,
                "judge_relevance": None, "judge_completeness": None, "judge_feasibility": None,
                "judge_clarity": None, "judge_agile_alignment": None, "judge_overall": None,
                "judge_strengths": None, "judge_weaknesses": None, "judge_verdict": None,
            })

        with open(ckpt_path, "w") as f:
            json.dump(model_results, f)
        print(f"  💾 Checkpoint saved: {ckpt_path}")
        all_results.extend(model_results)
        unload_model(model, tokenizer)

    # ── API (Gemini) models ────────────────────────────────────────────────
    for model_name, model_id in api_models.items():
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_{run_id}.json")
        if os.path.exists(ckpt_path):
            with open(ckpt_path) as f:
                model_results = json.load(f)
            print(f"\n♻️  {model_name}: loaded {len(model_results)} results from checkpoint")
            all_results.extend(model_results)
            continue

        raw_results = run_gemini_inference(dataset, model_id, model_name)

        model_results = []
        for raw in raw_results:
            metrics = compute_real_metrics(raw)
            model_results.append({
                "model": model_name, "sample_id": raw["sample_id"], "task_type": raw["task_type"],
                "input": raw["input"], "reference": raw["reference"], "response": raw["response"],
                "latency_s": raw["latency_s"], "tokens_gen": raw["tokens_gen"], "tokens_per_sec": raw["tokens_per_sec"],
                **metrics,
                "judge_relevance": None, "judge_completeness": None, "judge_feasibility": None,
                "judge_clarity": None, "judge_agile_alignment": None, "judge_overall": None,
                "judge_strengths": None, "judge_weaknesses": None, "judge_verdict": None,
            })

        with open(ckpt_path, "w") as f:
            json.dump(model_results, f)
        print(f"  💾 Checkpoint saved: {ckpt_path}")
        all_results.extend(model_results)

    return all_results, run_id


print("✅ Benchmark runner ready (local + API dispatch)")

✅ Benchmark runner ready (local + API dispatch)


In [13]:
print("🚀 Starting REAL-data benchmark...")
print(f"   Local models : {list(MODELS.keys())}")
print(f"   API models   : {list(API_MODELS.keys())}")
print(f"   Samples      : {len(REAL_BENCHMARK_DATASET)} (50 per task type)")
print()

wall_start = time.time()

# Phase 1 — inference (local models + Gemini)
# 4 base models × 200 samples (checkpoints)
all_results, run_id = run_inference_phase(REAL_BENCHMARK_DATASET, MODELS, API_MODELS)

# Phase 2 — judge
all_results = run_judge_phase(all_results)

# Save full results
df_results = pd.DataFrame(all_results)
output_csv = f"real_benchmark_results_{run_id}.csv"
df_results.to_csv(output_csv, index=False)

elapsed = (time.time() - wall_start) / 3600
print(f"\n✅ Real-data benchmark complete!")
print(f"   Wall time : {elapsed:.2f} hours")
print(f"   Results   : {output_csv} ({len(df_results)} rows)")

🚀 Starting REAL-data benchmark...
   Local models : ['Qwen3-8B', 'GLM-Z1-9B', 'Llama-3.1-8B']
   API models   : ['Gemini-3.5-Flash-Lite']
   Samples      : 200 (50 per task type)


♻️  Qwen3-8B: loaded 200 results from checkpoint

♻️  GLM-Z1-9B: loaded 200 results from checkpoint

⏳ Loading Llama-3.1-8B
   HF ID : meta-llama/Llama-3.1-8B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (10 outputs left out)

Gemini-3.5-Flash-Lite: 100%|██████████████████| 200/200 [23:12<00:00,  6.96s/it]



✅ Gemini inference done
  💾 Checkpoint saved: checkpoints_real/Gemini-3.5-Flash-Lite_real_001.json

🧑‍⚖️  Parallel judge | 800 samples | 3 keys


Judge: 100%|██████████████████████████████████| 800/800 [19:10<00:00,  1.44s/it]


✅ Judge complete | errors: 155 / 800

✅ Real-data benchmark complete!
   Wall time : 0.88 hours
   Results   : real_benchmark_results_real_001.csv (800 rows)


## Cells 9–12 — Per-track analysis (ran **before** the FT model was added)

The next four cells were run in Colab **before Cell 13**, so their outputs show only the 4 base models and match
Notebook 5. Two of them stop with errors:

* **Cell 10** uses an older `wilcoxon_pairwise` (no `dropna=False`, no dtype coercion): on `hours_mape` the
  pivot table has no column for the local models (all their values are NaN) → `KeyError: 'Qwen3-8B'`. Notebook 5's
  version of this cell handles that case.
* **Cell 12** fails at the judge radar (`TypeError … dtype->object`) for the same reason as in Notebook 5.

For the 5-model versions of these tables, tests and charts see **Notebook 7**.

In [14]:
df_results = pd.read_csv(output_csv) if "output_csv" in dir() else pd.read_csv("real_benchmark_results_real_001.csv")  # 4-model CSV at this point (FT not merged yet)

print("="*70)
print("TASK DESIGN — Schema Compliance · Priority-Match · Judge")
print("="*70)
td = df_results[df_results["task_type"] == "task_design"]
print(td.groupby("model")[["auto_schema_compliance", "priority_match", "judge_overall"]].mean().round(3))

print("\n" + "="*70)
print("TASK ALLOCATION — Assignee Match · Hours MAPE · Judge")
print("="*70)
ta = df_results[df_results["task_type"] == "task_allocation"]
print(ta.groupby("model")[["assignee_exact_match", "hours_mape", "judge_overall"]].mean().round(3))

print("\n" + "="*70)
print("SPRINT PLANNING — Task-Set F1 · Scope-Size MAPE · Judge")
print("="*70)
sp = df_results[df_results["task_type"] == "sprint_planning"]
print(sp.groupby("model")[["taskset_precision", "taskset_recall", "taskset_f1", "scope_mape", "judge_overall"]].mean().round(3))

print("\n" + "="*70)
print("RESOURCE PLANNING — Velocity MAPE (low-confidence) · Schema · Judge")
print("="*70)
rp = df_results[df_results["task_type"] == "resource_planning"]
print(rp.groupby("model")[["velocity_mape", "auto_schema_compliance", "judge_overall"]].mean().round(3))

TASK DESIGN — Schema Compliance · Priority-Match · Judge
                       auto_schema_compliance priority_match  judge_overall
model                                                                      
GLM-Z1-9B                                0.24           0.08          1.746
Gemini-3.5-Flash-Lite                    1.00           0.48          2.823
Llama-3.1-8B                             0.82            0.4          2.690
Qwen3-8B                                 0.98           0.48          2.128

TASK ALLOCATION — Assignee Match · Hours MAPE · Judge
                      assignee_exact_match  hours_mape  judge_overall
model                                                                
GLM-Z1-9B                             0.02         NaN          1.138
Gemini-3.5-Flash-Lite                 0.24      30.732          2.156
Llama-3.1-8B                           0.0         NaN          1.139
Qwen3-8B                               0.0         NaN          1.260

SPRINT PLAN

In [ ]:
from scipy import stats
from itertools import combinations


# Older version — breaks when one model has no valid values (see note above).
def wilcoxon_pairwise(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    models = df["model"].unique()
    rows = []
    for m1, m2 in combinations(models, 2):
        pivot = (df[df["model"].isin([m1, m2])]
                 .pivot_table(index="sample_id", columns="model", values=metric)
                 .dropna())
        if len(pivot) < 10:
            print(f"⚠️  Too few paired samples for {m1} vs {m2} on {metric}")
            continue
        stat, p = stats.wilcoxon(pivot[m1], pivot[m2])
        diff = pivot[m1].mean() - pivot[m2].mean()
        rows.append({
            "Model A": m1, "Model B": m2,
            "Mean A": round(pivot[m1].mean(), 4), "Mean B": round(pivot[m2].mean(), 4),
            "Δ (A–B)": round(diff, 4), "p-value": round(p, 4),
            "Significant": "✅ Yes" if p < 0.05 else "❌ No",
            "Better": m1 if diff > 0 else m2,
        })
    return pd.DataFrame(rows)


TRACK_PRIMARY_METRIC = {
    "task_design": "judge_overall",
    "task_allocation": "hours_mape",       # lower is better — note when reading Δ  # only Gemini has hours_mape → KeyError below
    "sprint_planning": "taskset_f1",
    "resource_planning": "judge_overall",  # velocity_mape too sparse/low-confidence for a clean test
}

for task_type, metric in TRACK_PRIMARY_METRIC.items():
    sub = df_results[df_results["task_type"] == task_type]
    print("="*70)
    print(f"{task_type.upper()} — metric: {metric}")
    print("="*70)
    wdf = wilcoxon_pairwise(sub, metric)
    if not wdf.empty:
        print(wdf.to_string(index=False))
    print()

TASK_DESIGN — metric: judge_overall
     Model A               Model B  Mean A  Mean B  Δ (A–B)  p-value Significant                Better
    Qwen3-8B             GLM-Z1-9B  2.1000  1.7162   0.3838   0.0062       ✅ Yes              Qwen3-8B
    Qwen3-8B          Llama-3.1-8B  2.1160  2.6080  -0.4920   0.0005       ✅ Yes          Llama-3.1-8B
    Qwen3-8B Gemini-3.5-Flash-Lite  2.2042  2.7958  -0.5917   0.0017       ✅ Yes Gemini-3.5-Flash-Lite
   GLM-Z1-9B          Llama-3.1-8B  1.7071  2.7000  -0.9929   0.0000       ✅ Yes          Llama-3.1-8B
   GLM-Z1-9B Gemini-3.5-Flash-Lite  1.7607  2.8107  -1.0500   0.0000       ✅ Yes Gemini-3.5-Flash-Lite
Llama-3.1-8B Gemini-3.5-Flash-Lite  2.6526  2.7105  -0.0579   0.7575        ❌ No Gemini-3.5-Flash-Lite

TASK_ALLOCATION — metric: hours_mape
⚠️  Too few paired samples for Qwen3-8B vs GLM-Z1-9B on hours_mape
⚠️  Too few paired samples for Qwen3-8B vs Llama-3.1-8B on hours_mape


KeyError: 'Qwen3-8B'

In [16]:
# Same per-track paper tables as Notebook 5 (4 models).
def export_track_table(task_type: str, cols: list, name: str):
    sub = df_results[df_results["task_type"] == task_type]
    summary = sub.groupby("model")[cols].mean().round(3)
    print(f"\n{name}")
    print(summary.to_string())
    summary.to_csv(f"paper_table_{task_type}.csv")
    print(summary.to_latex(float_format="%.3f", caption=name, label=f"tab:{task_type}"))
    return summary


export_track_table("task_design", ["auto_schema_compliance", "priority_match", "judge_overall"],
                    "Real-World Task Design (N per model = 50, reference-free)")

export_track_table("task_allocation", ["assignee_exact_match", "hours_mape", "judge_overall"],
                    "Real-World Task Allocation (N per model = 50, real assignee/hours ground truth)")

export_track_table("sprint_planning", ["taskset_f1", "scope_mape", "judge_overall"],
                    "Real-World Sprint Planning (N per model = 50, fixVersion-as-sprint)")

export_track_table("resource_planning", ["velocity_mape", "auto_schema_compliance", "judge_overall"],
                    "Real-World Resource Planning (N per model = 50, velocity = issues/release)")


Real-World Task Design (N per model = 50, reference-free)
                       auto_schema_compliance priority_match  judge_overall
model                                                                      
GLM-Z1-9B                                0.24           0.08          1.746
Gemini-3.5-Flash-Lite                    1.00           0.48          2.823
Llama-3.1-8B                             0.82            0.4          2.690
Qwen3-8B                                 0.98           0.48          2.128
\begin{table}
\caption{Real-World Task Design (N per model = 50, reference-free)}
\label{tab:task_design}
\begin{tabular}{lrlr}
\toprule
 & auto_schema_compliance & priority_match & judge_overall \\
model &  &  &  \\
\midrule
GLM-Z1-9B & 0.240 & 0.080 & 1.746 \\
Gemini-3.5-Flash-Lite & 1.000 & 0.480 & 2.823 \\
Llama-3.1-8B & 0.820 & 0.400 & 2.690 \\
Qwen3-8B & 0.980 & 0.480 & 2.128 \\
\bottomrule
\end{tabular}
\end{table}


Real-World Task Allocation (N per model = 50, real assign

,velocity_mape,auto_schema_compliance,judge_overall
model,,,
GLM-Z1-9B,0.001,0.06,1.251
Gemini-3.5-Flash-Lite,0.000,1.00,1.721
Llama-3.1-8B,0.000,0.36,1.730
Qwen3-8B,0.400,1.00,1.550


In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

MODEL_COLORS = {
    "Qwen3-8B":              "#2196F3",
    "Llama-3.1-8B":          "#4CAF50",
    "GLM-Z1-9B":             "#FF5722",
    "Gemini-3.5-Flash-Lite": "#9C27B0",
}
BG = "#0d1117"  # dark theme (Notebook 5 switched to white)


def fig_setup(w=10, h=6):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.tick_params(colors="white")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["bottom", "left"]].set_color("#333")
    return fig, ax


def plot_track_bars(df: pd.DataFrame, task_type: str, metrics: list, labels: list,
                     title: str, filename: str, higher_is_better: list = None):
    """Grouped bar chart for one track's metrics, one bar-group per model."""
    sub = df[df["task_type"] == task_type]
    agg = sub.groupby("model")[metrics].mean()
    models = [m for m in MODEL_COLORS if m in agg.index]  # keep consistent order

    fig, ax = fig_setup(10, 5.5)
    n_metrics = len(metrics)
    n_models  = len(models)
    bar_w     = 0.8 / n_models
    x         = np.arange(n_metrics)

    for i, model in enumerate(models):
        vals = agg.loc[model, metrics].values
        offset = (i - (n_models - 1) / 2) * bar_w
        ax.bar(x + offset, vals, width=bar_w, label=model,
               color=MODEL_COLORS.get(model, "gray"))

    ax.set_xticks(x)
    ax.set_xticklabels(labels, color="white", fontsize=10)
    ax.set_title(title, color="white", fontsize=13, pad=15)
    ax.legend(loc="upper right", labelcolor="white", facecolor="#111", edgecolor="#444", fontsize=8)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, facecolor=BG)
    plt.show()
    print(f"💾 saved {filename}")


def plot_judge_radar(df: pd.DataFrame):
    """Judge-score radar — the one metric computed identically across all four tracks."""
    dims   = ["judge_relevance", "judge_completeness", "judge_feasibility",
              "judge_clarity", "judge_agile_alignment"]
    labels = ["Relevance", "Completeness", "Feasibility", "Clarity", "Agile\nAlign"]
    avail  = [d for d in dims if d in df.columns]
    if not avail:
        print("No judge scores available yet.")
        return

    agg = df.groupby("model")[avail].mean()
    N   = len(avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.grid(color="#333", linewidth=0.5)
    ax.spines["polar"].set_color("#444")

    for model in agg.index:
        vals = agg.loc[model].tolist() + [agg.loc[model, avail[0]]]
        c = MODEL_COLORS.get(model, "gray")
        ax.plot(angles, vals, "o-", lw=2, color=c, label=model)
        ax.fill(angles, vals, alpha=0.12, color=c)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, color="white", fontsize=10)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1", "2", "3", "4", "5"], color="#666", fontsize=8)
    ax.set_title("LLM-as-Judge Scores Across All 4 Models (1–5)", color="white", fontsize=13, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1),
              labelcolor="white", facecolor="#111", edgecolor="#444")
    plt.tight_layout()
    plt.savefig("radar_judge_real.png", dpi=150, facecolor=BG)
    plt.show()
    print("💾 saved radar_judge_real.png")


# ── Generate all charts ────────────────────────────────────────────────────

plot_track_bars(
    df_results, "task_design",
    metrics=["auto_schema_compliance", "priority_match", "judge_overall"],
    labels=["Schema\nCompliance", "Priority\nMatch", "Judge/5"],
    title="Real-World Task Design", filename="chart_task_design.png",
)

plot_track_bars(
    df_results, "task_allocation",
    metrics=["assignee_exact_match", "hours_mape", "judge_overall"],
    labels=["Assignee\nExact Match", "Hours\nMAPE (lower=better)", "Judge/5"],
    title="Real-World Task Allocation", filename="chart_task_allocation.png",
)

plot_track_bars(
    df_results, "sprint_planning",
    metrics=["taskset_f1", "scope_mape", "judge_overall"],
    labels=["Task-Set F1", "Scope MAPE\n(lower=better)", "Judge/5"],
    title="Real-World Sprint Planning", filename="chart_sprint_planning.png",
)

plot_track_bars(
    df_results, "resource_planning",
    metrics=["velocity_mape", "auto_schema_compliance", "judge_overall"],
    labels=["Velocity MAPE\n(low-confidence)", "Schema\nCompliance", "Judge/5"],
    title="Real-World Resource Planning", filename="chart_resource_planning.png",
)

plot_judge_radar(df_results)  # TypeError: judge columns are object dtype after the CSV round-trip

💾 saved chart_task_design.png
💾 saved chart_task_allocation.png
💾 saved chart_sprint_planning.png
💾 saved chart_resource_planning.png


TypeError: agg function failed [how->mean,dtype->object]

---
## Cell 13 — NEW: Evaluate `Qwen3-8B-FT` on the real dataset and merge it as the 5th model

The cell is written as five logical sub-cells (A–E):

| Part | What it does | Reuses from earlier cells |
|---|---|---|
| **A** | config: base model, adapter **pulled from the HF Hub**, display name, batch 32 | — |
| **B** | loader (bf16 base + LoRA → `merge_and_unload` → `torch.compile`), prompt builder with `enable_thinking=False`, batched generation | `MAX_NEW_TOKENS`, `TEMPERATURE`, `DO_SAMPLE` |
| **C** | run inference on the 200 real samples → `compute_real_metrics` → checkpoint `checkpoints_real/Qwen3-8B-FT_real_001.json` | `REAL_BENCHMARK_DATASET`, `compute_real_metrics` |
| **D** | judge the 200 FT answers | `run_judge_phase` |
| **E** | merge into `real_benchmark_results_real_001.csv`, replacing any earlier FT rows (safe to re-run) | `output_csv` |

**Why a separate loader?** The FT model is evaluated the way it was in Notebook 4 — merged bf16 weights,
Flash-Attention 2, `torch.compile`, batch 32 — not with the 4-bit loader used for the base SLMs. Quality should be
very close, but **latency and throughput are not like-for-like** with the 4-bit base models.

**Prompting difference:** the FT prompt uses Qwen's official `enable_thinking=False` switch, while base Qwen3-8B in
Notebook 5 used the `/no_think` string trick (base answers still begin with an empty `<think></think>` block).

In [12]:
# =============================================================================
# Qwen3-8B-FT on the REAL-WORLD dataset — paste these as NEW CELLS at the end
# of `real_data_slm_benchmark_v6.ipynb`, AFTER Cell 18 (the main benchmark run).
#
# Reuses, unchanged, from the v6 notebook already in memory:
#   - REAL_BENCHMARK_DATASET   (Cell 3)
#   - compute_real_metrics()   (Cell 5)
#   - run_judge_phase()        (Cell 6)
#   - CHECKPOINT_DIR, MAX_NEW_TOKENS, TEMPERATURE, DO_SAMPLE  (Cell 2)
#   - output_csv / run_id      (Cell 8, "real_001")
#
# It does NOT touch Qwen3-8B / Llama-3.1-8B / GLM-Z1-9B / Gemini results —
# it only adds a 5th model's rows and merges them into the same CSV.
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────
# New Cell A — FT config (mirrors notebook 2's Cell 2, adapted for v6 paths)
# ─────────────────────────────────────────────────────────────────────────
import os

FT_BASE_MODEL_ID  = "Qwen/Qwen3-8B"
FT_ADAPTER_PATH   = "arjunverma2004/Qwen3-8B-PM-LoRA"   # HF Hub repo — swap for a local dir if you'd rather load from disk  # adapter downloaded from the Hugging Face Hub
FT_NAME           = "Qwen3-8B-FT"
FT_INFERENCE_BATCH_SIZE = 32                            # matches nb2 (A100, merged bf16)  # bigger batch than the 4-bit base models (12)

print(f"✅ FT config loaded — {FT_NAME} from {FT_ADAPTER_PATH}")


# ─────────────────────────────────────────────────────────────────────────
# New Cell B — FT model loader + prompt builder + batched inference
# Identical mechanics to notebook 2's Cell 9 (bf16 merge, no 4-bit, Flash
# Attention 2, torch.compile) — NOT the 4-bit loader used for the other
# three local models, since that's how Qwen3-8B-FT was evaluated "previously".
# ─────────────────────────────────────────────────────────────────────────
import torch
import time
import gc
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


# bf16 base + LoRA adapter, merged and compiled (same as Notebook 4, Cell 9).
def load_finetuned_model(base_model_id: str, adapter_path: str):
    print(f"\n{'='*60}")
    print(f"⏳ Loading fine-tuned model")
    print(f"   Base   : {base_model_id}")
    print(f"   Adapter: {adapter_path}")
    print(f"{'='*60}")
    t0 = time.time()

    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path, token=os.environ.get("HF_TOKEN"), trust_remote_code=True,  # tokenizer saved with the adapter
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,  # full bf16 — no 4-bit for the FT model
        device_map="auto",
        token=os.environ.get("HF_TOKEN"),
        trust_remote_code=True,
        attn_implementation="flash_attention_2",
    )

    print("🔀 Merging LoRA adapter...")
    model = PeftModel.from_pretrained(base, adapter_path, token=os.environ.get("HF_TOKEN"))
    model = model.merge_and_unload()  # fold LoRA into the base weights → no adapter overhead
    model.eval()

    if hasattr(torch, "compile"):
        print("⚡ Compiling model with torch.compile...")
        model = torch.compile(model, mode="reduce-overhead")  # faster repeated decoding

    load_time = time.time() - t0
    vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"✅ {FT_NAME} ready | Load: {load_time:.1f}s | VRAM: {vram:.1f} GB")
    return tokenizer, model


# Same system prompt as every other model; thinking disabled the official way.
def build_prompt_ft(sample: dict, tokenizer) -> str:
    messages = [
        {"role": "system", "content": (
            "You are an expert Agile project manager. "
            "When asked to output JSON, respond ONLY with valid JSON — "
            "no markdown fences, no explanations, no preamble."
        )},
        {"role": "user", "content": sample["input"]},
    ]
    try:
        # enable_thinking=False — same as notebook 2, disables Qwen3 CoT properly
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,  # Qwen3 chat-template switch
        )
    except Exception:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )


# Batched generation, same result schema as Notebook 5 (incl. _meta).
def run_batched_inference_ft(tokenizer, model, dataset: list,
                              batch_size: int = FT_INFERENCE_BATCH_SIZE) -> list:
    results   = []
    prompts   = [build_prompt_ft(s, tokenizer) for s in dataset]
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    total_tokens = 0
    wall_t0 = time.time()

    print(f"\n🚀 {FT_NAME} inference: {len(dataset)} samples | batch_size={batch_size} | {n_batches} batches")

    for bi in tqdm(range(n_batches), desc=FT_NAME, ncols=80):
        s, e = bi * batch_size, min((bi + 1) * batch_size, len(dataset))
        batch_prompts = prompts[s:e]
        batch_samples = dataset[s:e]

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=1024)  # ⚠️ same 1,024-token prompt truncation as the base models
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[1]

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
                do_sample=DO_SAMPLE, pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.05, use_cache=True,  # same penalty as the base models; KV cache on
            )
        batch_latency = time.time() - t0

        for sample, output_ids in zip(batch_samples, outputs):
            generated_ids = output_ids[input_len:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            n_tokens = int(generated_ids.shape[0])  # ⚠️ padded batch length, not this answer's length → tok/s inflated
            total_tokens += n_tokens

            results.append({
                "sample_id": sample["id"], "task_type": sample["task_type"],
                "input": sample["input"], "reference": sample["reference_output"],
                "response": response,
                "latency_s": round(batch_latency / len(batch_samples), 2),
                "tokens_gen": n_tokens,
                "tokens_per_sec": round(n_tokens / (batch_latency / len(batch_samples)), 1),
                # carry through real-track metadata for scoring — same as v6's run_batched_inference
                "_meta": {k: v for k, v in sample.items() if k not in ("id", "task_type", "input", "reference_output")},
            })

    elapsed = time.time() - wall_t0
    print(f"\n✅ {FT_NAME} inference done | {elapsed/60:.1f} min | {total_tokens:,} tokens | {total_tokens/elapsed:.0f} tok/s avg")
    return results


# Free the GPU before judging.
def unload_ft_model(model, tokenizer):
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"🗑️  {FT_NAME} unloaded | VRAM now: {torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0:.1f} GB")


print(f"✅ {FT_NAME} inference engine ready (bf16 merge + Flash Attention 2 + torch.compile)")


# ─────────────────────────────────────────────────────────────────────────
# New Cell C — Run FT inference on the real dataset, score with the SAME
# real-track metrics engine (compute_real_metrics from v6 Cell 5), and
# checkpoint exactly like v6's run_inference_phase does for the other models.
# ─────────────────────────────────────────────────────────────────────────
# Part C — same run id → checkpoint sits next to the other models'
run_id = "real_001"   # must match the run_id used in v6 Cell 7/8
ft_ckpt_path = os.path.join(CHECKPOINT_DIR, f"{FT_NAME}_{run_id}.json")

if os.path.exists(ft_ckpt_path):  # resume support
    with open(ft_ckpt_path) as f:
        ft_model_results = json.load(f)
    print(f"♻️  {FT_NAME}: loaded {len(ft_model_results)} results from checkpoint")
else:
    tokenizer_ft, model_ft = load_finetuned_model(FT_BASE_MODEL_ID, FT_ADAPTER_PATH)
    raw_results_ft = run_batched_inference_ft(tokenizer_ft, model_ft, REAL_BENCHMARK_DATASET)
    unload_ft_model(model_ft, tokenizer_ft)

    ft_model_results = []
    for raw in raw_results_ft:
        metrics = compute_real_metrics(raw)   # <- same scorer used for Qwen/Llama/GLM/Gemini  # identical scoring to the 4 base models
        ft_model_results.append({
            "model": FT_NAME, "sample_id": raw["sample_id"], "task_type": raw["task_type"],
            "input": raw["input"], "reference": raw["reference"], "response": raw["response"],
            "latency_s": raw["latency_s"], "tokens_gen": raw["tokens_gen"], "tokens_per_sec": raw["tokens_per_sec"],
            **metrics,
            "judge_relevance": None, "judge_completeness": None, "judge_feasibility": None,
            "judge_clarity": None, "judge_agile_alignment": None, "judge_overall": None,
            "judge_strengths": None, "judge_weaknesses": None, "judge_verdict": None,
        })

    with open(ft_ckpt_path, "w") as f:
        json.dump(ft_model_results, f)
    print(f"  💾 Checkpoint saved: {ft_ckpt_path}")

print(f"\n✅ {FT_NAME}: {len(ft_model_results)} samples scored on real-track metrics")


# ─────────────────────────────────────────────────────────────────────────
# New Cell D — Judge phase for the FT results (reuses v6's run_judge_phase,
# same Inception/mercury-2 keys, same reference-light real-sample prompt)
# ─────────────────────────────────────────────────────────────────────────
# Part D — same judge as the base models.
ft_model_results = run_judge_phase(ft_model_results)

with open(ft_ckpt_path, "w") as f:
    json.dump(ft_model_results, f)
print(f"💾 Checkpoint updated with judge scores: {ft_ckpt_path}")


# ─────────────────────────────────────────────────────────────────────────
# New Cell E — Merge into the SAME results CSV as the other 4 models
# (real_benchmark_results_real_001.csv from v6 Cell 8). Any prior
# Qwen3-8B-FT rows in that file are replaced, not duplicated, so this
# cell is safe to re-run.
# ─────────────────────────────────────────────────────────────────────────
# Part E — merge into the shared results file.
output_csv = f"real_benchmark_results_{run_id}.csv"

df_existing = pd.read_csv(output_csv) if os.path.exists(output_csv) else pd.DataFrame()
df_ft = pd.DataFrame(ft_model_results)

if not df_existing.empty:
    df_existing = df_existing[df_existing["model"] != FT_NAME]  # drop old FT rows → re-running never duplicates
    df_combined = pd.concat([df_existing, df_ft], ignore_index=True)
else:
    df_combined = df_ft

df_combined.to_csv(output_csv, index=False)
df_results = df_combined  # so downstream analysis cells (v6 Cell 9 onward) pick it up  # so later analysis cells see all 5 models

print(f"\n✅ Merged. {output_csv} now has {len(df_combined)} rows across models: "
      f"{sorted(df_combined['model'].unique())}")

✅ FT config loaded — Qwen3-8B-FT from arjunverma2004/Qwen3-8B-PM-LoRA
✅ Qwen3-8B-FT inference engine ready (bf16 merge + Flash Attention 2 + torch.compile)

⏳ Loading fine-tuned model
   Base   : Qwen/Qwen3-8B
   Adapter: arjunverma2004/Qwen3-8B-PM-LoRA


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

🔀 Merging LoRA adapter...
⚡ Compiling model with torch.compile...
✅ Qwen3-8B-FT ready | Load: 6.6s | VRAM: 32.8 GB

🚀 Qwen3-8B-FT inference: 200 samples | batch_size=32 | 7 batches


Qwen3-8B-FT: 100%|████████████████████████████████| 7/7 [03:29<00:00, 29.95s/it]



✅ Qwen3-8B-FT inference done | 3.5 min | 78,072 tokens | 372 tok/s avg
🗑️  Qwen3-8B-FT unloaded | VRAM now: 16.4 GB
  💾 Checkpoint saved: checkpoints_real/Qwen3-8B-FT_real_001.json

✅ Qwen3-8B-FT: 200 samples scored on real-track metrics

🧑‍⚖️  Parallel judge | 200 samples | 3 keys


Judge: 100%|██████████████████████████████████| 200/200 [07:30<00:00,  2.25s/it]


✅ Judge complete | errors: 55 / 200
💾 Checkpoint updated with judge scores: checkpoints_real/Qwen3-8B-FT_real_001.json

✅ Merged. real_benchmark_results_real_001.csv now has 1000 rows across models: ['GLM-Z1-9B', 'Gemini-3.5-Flash-Lite', 'Llama-3.1-8B', 'Qwen3-8B', 'Qwen3-8B-FT']


### 📊 Reading the output

* Load 6.6 s (weights cached), **32.8 GB VRAM** (bf16 + compile). Inference: **200 samples in 3.5 min,
  78,072 tokens, 372 tok/s**.
* `✅ Judge complete | errors: 55 / 200` → 145 FT answers have a judge score.
* The merged file has **1,000 rows** (5 models × 200). It is saved in the repository as
  `Finetunning/result with RD qwen-ft/ft-real_benchmark_results_real_001.csv`.

### What the merged results show (computed in Notebook 7)

* **Task design improves the most:** judge 2.13 → **2.68**, schema 0.98 → 1.00 — the FT model now matches Llama and
  is close to Gemini on the one real track whose format matches the synthetic training data.
* **Sprint planning breaks:** 45 of 50 FT answers stop mid-JSON at the 600-token limit and none parse → task-set F1 **0.00**
  (base 0.27) and judge 1.86 (base 2.41). The FT model writes the synthetic sprint format (a rich object per task)
  instead of the list of titles + `scope_size` the real prompt asks for.
* **Task allocation is unusable for every local model, FT included** — the prompt is cut at 1,024 tokens and
  no FT answer starts with `{` (40 % begin with text copied straight from the prompt's roster).
* **Speed:** 1.05 s per sample vs 3.66 s for base Qwen (but bf16/A100 vs 4-bit/L40S).
* **Throughput is overstated.** `n_tokens = generated_ids.shape[0]` is the length of the *padded* batch output, so
  every sample in a batch gets the batch's longest length (only 6 distinct `tokens_gen` values in 200 rows). The base
  engine counts real tokens. The FT tok/s figures (372 here, 446 in Notebook 4) are therefore inflated; latency per
  sample is fine.

### ⚠️ Reviewer notes

1. **Synthetic gains did not transfer.** On synthetic data the FT model gained +1.16 judge points; on real data the
   overall judge is flat (1.81 → 1.83). That is a key finding for the paper — the adapter learned the synthetic
   output formats.
2. To test the FT model fairly on sprint planning, either raise `MAX_NEW_TOKENS` or include real-format sprint
   examples in training.
3. Re-run task allocation for all local models with a longer `max_length` before reporting that track.